In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [11]:
words = open('names.txt', 'r').read().splitlines()
print(words[:8], f'... len: {len(words)}')

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'] ... len: 32033


In [20]:
chrs = sorted(list(set(''.join(words))))
stoi = {s : i+1 for i, s in enumerate(chrs)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}

In [22]:
# dataset
block_size = 3
X, Y = [], []
for w in words[:5]:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [87]:
X.shape

torch.Size([32, 3])

In [ ]:
X[:5] # [0, 5, 13] is gonna be one of our input, so all of them will have their own coordinates(this is called embadded)

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])

In [83]:
C = torch.randn((27, 2)) # this contains all coordinates (vectors) of our embadded characters (from '.' - 0 and 'a' - 1 to 'z' - 26)

In [131]:
""" 
C[X[:5]] replaces every index in X[:5] with its row from C (its embedding vector)
  shape: (5, 3, 2)
    5 - examples (rows of X[:5])
    3 - characters in each example (context length)
    2 - size of each embedding vector 
"""
e = C[X[:5]] # we get the same result, but now every number in a row has its own embedded value
# so to store this 2d arrays we need one more demension: 5 - rows from the original array | 3 - lenghts of the rows in the original array | 2 - their embedded values
# thus every number gets its embaddened magnitude 
e

tensor([[[ 1.0916,  0.1629],
         [ 1.0916,  0.1629],
         [ 1.0916,  0.1629]],

        [[ 1.0916,  0.1629],
         [ 1.0916,  0.1629],
         [-0.7209, -0.1386]],

        [[ 1.0916,  0.1629],
         [-0.7209, -0.1386],
         [ 0.8775, -1.6749]],

        [[-0.7209, -0.1386],
         [ 0.8775, -1.6749],
         [ 0.8775, -1.6749]],

        [[ 0.8775, -1.6749],
         [ 0.8775, -1.6749],
         [-0.5331,  1.8900]]])

In [141]:
e[:, 0, :] # C[X[:5][:, 0, :]

tensor([[ 1.0916,  0.1629],
        [ 1.0916,  0.1629],
        [ 1.0916,  0.1629],
        [-0.7209, -0.1386],
        [ 0.8775, -1.6749]])

In [142]:
torch.unbind(C[X[:5]], 1) # [:, 0, :] & [:, 1, :] & [:, 2, :]

(tensor([[ 1.0916,  0.1629],
         [ 1.0916,  0.1629],
         [ 1.0916,  0.1629],
         [-0.7209, -0.1386],
         [ 0.8775, -1.6749]]),
 tensor([[ 1.0916,  0.1629],
         [ 1.0916,  0.1629],
         [-0.7209, -0.1386],
         [ 0.8775, -1.6749],
         [ 0.8775, -1.6749]]),
 tensor([[ 1.0916,  0.1629],
         [-0.7209, -0.1386],
         [ 0.8775, -1.6749],
         [ 0.8775, -1.6749],
         [-0.5331,  1.8900]]))

`W1 = torch.randn((6, 100))` -- this is gonna be our first hidden layer; \
therefore the problem is that C[X[:5]].shape = (5, 3, 2), so we need to somehow concatenate 3 and 2 together to make .shape equal to (5, 6)

In [ ]:
torch.cat([C[X[:5]][:, 0, :], C[X[:5]][:, 1, :], C[X[:5]][:, 2, :]], 1) # always needs to be changed if we change the block_size variable
# or
torch.cat(torch.unbind(e, 1), 1) # universal approach

tensor([[ 1.0916,  0.1629,  1.0916,  0.1629,  1.0916,  0.1629],
        [ 1.0916,  0.1629,  1.0916,  0.1629, -0.7209, -0.1386],
        [ 1.0916,  0.1629, -0.7209, -0.1386,  0.8775, -1.6749],
        [-0.7209, -0.1386,  0.8775, -1.6749,  0.8775, -1.6749],
        [ 0.8775, -1.6749,  0.8775, -1.6749, -0.5331,  1.8900]])

But the concatenate operation is very inefficient. \
The solution is to use PyTorch's .view() function.

In [143]:
C[X[:5]].view(5, 6)

tensor([[ 1.0916,  0.1629,  1.0916,  0.1629,  1.0916,  0.1629],
        [ 1.0916,  0.1629,  1.0916,  0.1629, -0.7209, -0.1386],
        [ 1.0916,  0.1629, -0.7209, -0.1386,  0.8775, -1.6749],
        [-0.7209, -0.1386,  0.8775, -1.6749,  0.8775, -1.6749],
        [ 0.8775, -1.6749,  0.8775, -1.6749, -0.5331,  1.8900]])

___

In [90]:
emb = C[X] # our firs layer
emb.shape

torch.Size([32, 3, 2])

In [137]:
# first hidden layer with tanh()
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
# torch.cat(torch.unbind(emb, 1), 1) 
hidden_layer = torch.tanh(emb.view(-1, 6) @ W1 + b1) # it is good to check broadcasting here
# (emb.view(-1, 6) @ W1).shape
# b1.shape

In [149]:
# output_layer
W2 = torch.randn((100, 27))
b2 = torch.rand(27)

In [157]:
# computing the output
logits = hidden_layer @ W2 + b2
# print(logits.shape)
# --- softmax ---
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
# ---------------
loss = -probs[torch.arange(32), Y].log().mean()
print(f'loss: {loss}')

loss: 16.6153507232666
